# 06 - Deploy

Turns an MLflow alias into a SageMaker endpoint.

### Resolution order

1. **MLflow alias → model version.** The source of truth for *which* model.
2. **The mirrored SageMaker model package**, if present → `ModelPackage.deploy()`.
   Preferred: the endpoint then references a registry object and keeps the usual
   SageMaker lineage.
3. **Fallback**: read `training_job_name` off the MLflow run, look up that job's
   `model.tar.gz`, and wrap it in the built-in XGBoost image. Used when the
   mirror has not appeared.

The deployment report tells you which path was taken (`source`).

### Endpoint naming

`player-churn-staging` and `player-churn-prod`, updated in place. Repeated
deploys do not leak endpoints — which the workshop's per-execution endpoints did.

In [1]:
import json
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)
os.environ["CHURN_PROJECT_ROOT"] = str(PROJECT_ROOT)

# Put this kernel's venv bin/ first on PATH so shell commands (dvc, ...) resolve to the
# venv, however Jupyter was launched.
import sys
os.environ["PATH"] = f"{Path(sys.executable).parent}{os.pathsep}{os.environ['PATH']}"
for line in (PROJECT_ROOT / ".env").read_text().strip().splitlines():
    key, _, value = line.partition("=")
    os.environ[key] = value

from churn.config import Settings, load_params

params = load_params()
settings = Settings.load(params)
print("region:", settings.region, "| bucket:", settings.bucket)

region: us-east-1 | bucket: sagemaker-us-east-1-063676201336


## 1. What does `@staging` point at?

In [2]:
from churn.registry import resolve_alias

staging_model = resolve_alias("staging", settings=settings, params=params)
print("model     :", staging_model.name, "v" + staging_model.version)
print("mlflow run:", staging_model.run_id)
print("sm package:", staging_model.sagemaker_model_package_arn or "(none - will use the MLflow artifact path)")

/Users/ritabanchowdhury/.venvs/churn-mlops/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


sagemaker.config INFO - Not applying SDK defaults from location: /usr/local/share/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /Users/ritabanchowdhury/Library/Application Support/sagemaker/config.yaml


/Users/ritabanchowdhury/.venvs/churn-mlops/lib/python3.11/site-packages/sagemaker/__init__.py:86: SageMakerV2DeprecationWarning: You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation()
You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


model     : player-churn-xgboost v2
mlflow run: 0eb9b0c6f641470abae33de9b38e890a
sm package: (none - will use the MLflow artifact path)


## 2. Deploy to staging

~6-8 minutes. `ml.m5.large`, no data capture — see `params.yaml` → `deploy.staging`.

In [3]:
from churn.deployment import deploy

staging = deploy(environment="staging", settings=settings, params=params)
print(f"\nendpoint {staging.endpoint_name} serving v{staging.model_version} (via {staging.source})")

/Users/ritabanchowdhury/.venvs/churn-mlops/lib/python3.11/site-packages/sagemaker/model.py:347: SageMakerV2DeprecationWarning: Model is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `ModelBuilder` (`from sagemaker.serve import ModelBuilder`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
Model is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `ModelBuilder` (`from sagemaker.serve import ModelBuilder`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


Creating endpoint player-churn-staging from player-churn-xgboost v2 (via mlflow_artifact)
-------!
endpoint player-churn-staging serving v2 (via mlflow_artifact)


## 3. Smoke test

Rows from the held-out test split, sent as CSV. The helper asserts the endpoint
returns scores in `[0, 1]`, which is the check CI relies on.

In [4]:
from churn.deployment import smoke_test

rows = "\n".join(Path("data/processed/test_x.csv").read_text().splitlines()[:5])
scores = smoke_test(staging.endpoint_name, settings.region, rows)
print("churn probabilities:", [round(s, 4) for s in scores])

/Users/ritabanchowdhury/.venvs/churn-mlops/lib/python3.11/site-packages/sagemaker/base_predictor.py:140: SageMakerV2DeprecationWarning: Predictor is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `sagemaker.core.resources.Endpoint`.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
Predictor is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `sagemaker.core.resources.Endpoint`.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


churn probabilities: [0.0825, 0.3781, 0.0888, 0.0133, 0.9369]


In [5]:
# Compare against the true labels for those same rows.
import pandas as pd

labels = pd.read_csv("data/processed/test_y.csv", header=None).squeeze().head(5).tolist()
pd.DataFrame({"probability": [round(s, 4) for s in scores], "actual": labels})

,probability,actual
0,0.0825,0
1,0.3781,0
2,0.0888,0
3,0.0133,0
4,0.9369,1


## 4. Promote to production

Moving `@production` also flips the mirrored SageMaker model package to
`Approved`, so Studio's registry and anything else reading model packages agrees
with MLflow.

In CI this is the **churn / deploy** workflow with `environment: production`,
which pauses on the `churn-production` GitHub Environment until a reviewer
approves. Doing it here, by hand, bypasses that gate — fine while you are
setting things up, not something to keep doing.

In [6]:
from churn.registry import ALIAS_PRODUCTION, promote

promoted = promote(
    version=staging_model.version,
    alias=ALIAS_PRODUCTION,
    settings=settings,
    params=params,
)
print(f"@production -> v{promoted.version}")

@production -> v2


In [7]:
# Production config differs: bigger instance, and data capture ON so Model
# Monitor has traffic to compare against the prediction baseline from nb 04.
print(json.dumps(params["deploy"]["production"], indent=2))

{
  "instance_type": "ml.m5.xlarge",
  "instance_count": 1,
  "enable_data_capture": true,
  "data_capture_sampling_percentage": 100
}


In [8]:
production = deploy(environment="production", settings=settings, params=params)
print(f"\nendpoint {production.endpoint_name} serving v{production.model_version}")

Creating endpoint player-churn-prod from player-churn-xgboost v2 (via mlflow_artifact)
------!
endpoint player-churn-prod serving v2


In [9]:
scores = smoke_test(production.endpoint_name, settings.region, rows)
print("production smoke test:", [round(s, 4) for s in scores])

production smoke test: [0.0825, 0.3781, 0.0888, 0.0133, 0.9369]


## 5. Batch scoring

Most churn work is a nightly batch, not a live endpoint. Batch Transform needs no
endpoint at all — it spins up, scores a file, and shuts down, so it costs nothing
when idle.

In [10]:
import sagemaker
from sagemaker.transformer import Transformer

from churn.aws import sagemaker_session

session = sagemaker_session(settings.region)

# Reuse the model object the endpoint was built from.
sm = __import__("boto3").client("sagemaker", region_name=settings.region)
endpoint_config = sm.describe_endpoint(EndpointName=production.endpoint_name)["EndpointConfigName"]
model_name = sm.describe_endpoint_config(EndpointConfigName=endpoint_config)["ProductionVariants"][0]["ModelName"]
print("model:", model_name)

transformer = Transformer(
    model_name=model_name,
    instance_count=1,
    instance_type="ml.m5.large",
    output_path=f"{settings.s3_base}/batch-output",
    assemble_with="Line",
    accept="text/csv",
    sagemaker_session=session,
)

/Users/ritabanchowdhury/.venvs/churn-mlops/lib/python3.11/site-packages/sagemaker/transformer.py:115: SageMakerV2DeprecationWarning: Transformer is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `sagemaker.core.shapes.TransformJob`.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
Transformer is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `sagemaker.core.shapes.TransformJob`.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


model: player-churn-v2-1793203984


In [11]:
# Score the baseline file (every player, no labels).
baseline_s3 = session.upload_data(
    "data/processed/baseline.csv",
    bucket=settings.bucket,
    key_prefix=f"{settings.bucket_prefix}/batch-input",
)
print("input:", baseline_s3)

transformer.transform(baseline_s3, content_type="text/csv", split_type="Line", wait=True)
print("output:", transformer.output_path)

input: s3://sagemaker-us-east-1-063676201336/player-churn/dvc-mlflow/batch-input/baseline.csv


INFO:sagemaker:Creating transform job with name: sagemaker-xgboost-2026-10-28-16-16-54-597


.........................../miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
[2026-10-28:16:21:30:INFO] No GPUs detected (normal if no gpus installed)
[2026-10-28:16:21:30:INFO] No GPUs detected (normal if no gpus installed)
[2026-10-28:16:21:30:INFO] nginx config: 
worker_processes auto;
daemon off;
pid /tmp/nginx.pid;
error_log  /dev/stderr;
worker_rlimit_nofile 4096;
events {
  worker_connections 2048;
}
http {
  include /etc/nginx/mime.types;
  default_type application/octet-stream;
  access_log /dev/stdout combined;
  upstream gunicorn {
    server unix:/tmp/gunicorn.sock;
  }
  server {
    listen 8080 deferred;
    client_max_body_size 0;
    keepalive_timeout 3;
    location ~ ^/(ping|invocat

In [12]:
scored = pd.read_csv(f"{transformer.output_path}/baseline.csv.out", header=None, names=["probability"])
print(f"{len(scored)} players scored")
print(scored.describe().round(4).to_string())
print(f"\npredicted churners at 0.5: {(scored['probability'] >= 0.5).sum()}")

INFO:aiobotocore.credentials:Found credentials in shared credentials file: ~/.aws/credentials


1640 players scored
       probability
count    1640.0000
mean        0.1233
std         0.2617
min         0.0017
25%         0.0126
50%         0.0126
75%         0.0825
max         0.9843

predicted churners at 0.5: 139


In [13]:
Path("reports/deployment.json").write_text(
    json.dumps(
        {
            "endpoint_name": production.endpoint_name,
            "environment": "production",
            "model_name": production.model_name,
            "model_version": production.model_version,
            "source": production.source,
            "smoke_test_scores": scores if isinstance(scores, list) else [],
        },
        indent=2,
        default=float,
    ),
    encoding="utf-8",
)
print("CLI equivalent (what CI runs):")
print("python -m churn.deployment --environment production")

CLI equivalent (what CI runs):
python -m churn.deployment --environment production


---
## Next

**`07-github-actions.ipynb`** -- wire the whole thing to GitHub so you stop
running notebooks by hand.

⚠️ Both endpoints are now billing by the hour. `99-cleanup.ipynb` removes them.

### What changed vs. the original workshop
Old notebook 07 depended on a Service Catalog project having provisioned two
CodePipelines, two CloudFormation stacks and two CodeCommit/GitHub repos. Deploy
was "approve a model version in the registry and wait". Here it is one
`python -m churn.deployment --environment production`, which the deploy workflow
runs behind a GitHub Environment gate. Endpoints are also updated in place
instead of created per execution.